### Introduction

This section is to get familiar with the data's format

In [28]:
import json

### LOAD JSON INTO PYTHON (probably a more scalable way to do this, but we're just testing right now)

json_path = "C:/Users/xnevi/Downloads/restaurants.menu_item_variations.json"

with open(json_path, encoding="utf-8") as f:
    item_list = json.load(f)
f.close()

In [29]:
### STORE INTO PYTHON DICT

d = {}
for item in item_list:
    store = item['restaurant_name']
    if(store in d.keys()):
        d[store].append(item)
    else:
        d[store] = [item]

In [30]:
### PRINT SET OF Chick-fil-A item categories

cfa_categories = set()
for item in d["Chick-fil-A"]:
    cfa_categories.add(item['category'])
print(cfa_categories)

{'Beverages', 'Sauces', 'Salad Dressings', 'Desserts', 'Cool Wraps', 'Salads', 'Classic Chicken', 'Breakfast', 'Sides', "Kid's Meal"}


In [31]:
### PRINT SET OF Taco Bell categories

tacobell_categories = set()
for item in d["Taco Bell"]:
    tacobell_categories.add(item['category'])
print(tacobell_categories)

{'Beverages', 'Nachos', 'Cantina Chicken', 'Tacos', 'Specialties', 'Burritos', 'Condiments & Sauces', 'Breakfast', 'Sides', 'Chalupas'}


### Re-categorization of items

We want to categorize each item into "Entrees", "Drinks", "Sides" or other categories like meal add-ons / components (open to suggestion).

This is a step I think is necessary for our meal-building (setting limits on how many drinks or sides in a suggested meal).

In [39]:
### GET THE TOTAL SET OF ALL THE CATEGORIES FROM ALL THE RESTAURANTS IN THE DATABASE

food_categories=set()
for store in d.keys():
    for item in d[store]:
        food_categories.add(item['category'])
#print(food_categories) ## UNCOMMENT THIS LINE TO SEE THE FULL SET OF FOOD CATEGORIES

### Proposing a Table Structure

Note that some of the columns have lists, or json objects as their values. PostgreSQL supports having text lists or json objects as column values, so this works for now. We may want to create other tables and structure the schema differently depending on our search needs.

In the following example, 'cuisine_type' is an example of a text list value, 'allergen_info' is an example of a json value.

In [38]:
### PRINT THE COLUMNS THAT EACH ENTRY HAS
column_labels = d["Taco Bell"][0].keys()
print(column_labels)

dict_keys(['item_id', 'menu_item_id', 'menu_item_name', 'restaurant_name', 'cuisine_type', 'macronutrient_profile', 'recipe_items', 'price', 'golden_ratio', 'ai_description', 'category', 'allergen_info', 'nutrition_info', 'menu_card_image'])


In [37]:
### PRINT EXAMPLE OF FULL ITEM ENTRY

d["Taco Bell"][0]

{'item_id': 'taco_bell_bean_burrito',
 'menu_item_id': 'item_bc872feac0cd4c038de6a4bc27b91eef',
 'menu_item_name': 'Bean Burrito',
 'restaurant_name': 'Taco Bell',
 'cuisine_type': ['Mexican', 'American', 'Fast Food'],
 'macronutrient_profile': ['High Carb',
  'Low Calorie',
  'High Sodium',
  'Low Sugar'],
 'recipe_items': {},
 'price': 1.29,
 'golden_ratio': 0.371,
 'ai_description': 'Savor the hearty blend of seasoned refried beans, zesty red sauce, and melted cheese wrapped in a warm flour tortilla for a satisfying bite.',
 'category': 'Burritos',
 'allergen_info': {'contains': ['gluten', 'milk', 'wheat'],
  'may_contain': [],
  'does_not_contain': ['egg',
   'fish',
   'MSG',
   'peanuts',
   'shellfish',
   'soy',
   'tree nuts'],
  'unknown': ['glutamates',
   'mustard',
   'nitrates',
   'seeds',
   'sesame',
   'sulfites'],
  'allergy_info': 'Allergy Information: a Taco Bell Bean Burrito contains gluten, milk and wheat. a Taco Bell Bean Burrito does not contain egg, fish, MSG,

### Categorization of items

**Proposed solution:** Use an LLM to re-categorize (add a column to) items.

*Method 1* -- Let's take the example of Taco Bell item categories:

```['Beverages', 'Nachos', 'Cantina Chicken', 'Tacos', 'Specialties', 'Burritos', 'Condiments & Sauces', 'Breakfast', 'Sides', 'Chalupas']```

An LLM can pretty easily determine that 'Beverages' fall into Drinks, 'Sides' fall into Sides, etc. If all the "broader" categories can be placed into Drinks, Sides, Entrees, and whatever else we decide on, this makes categorization a bit easier.


*Method 2* -- Let's say hypothetically there's a restaurant with a category "Snacks" that contains "Hot Dog" (Entree) and "Cheese Fries" (Side). We can't put all the items under "Snacks" into Entrees or all the items into Sides. In order to categorize these items, we may need to go item by item, feeding the LLM more information such as item name.
